# Train classification model

Imports

In [1]:
import torch
from torch import nn
import torchvision.transforms as transforms
import sys
import os
import torch.nn.functional as F

# Build an absolute path from this notebook's parent directory
src_module_path = os.path.abspath(os.path.join('..', 'src'))

# Add to sys.path if not already present
if src_module_path not in sys.path:
    sys.path.append(src_module_path)

from data import get_data, show_image
import config

Data Initialization

In [2]:
trainloader, testloader = get_data(
    "Mirabest",
    transforms.Compose([
        transforms.ToTensor(), # to range [0,1]
        transforms.Normalize([0.5], [0.5]) # 0 centers
        ]),
        config.CLASSIFICATION_MODEL["batch_size"])

print(next(iter(trainloader))[0].shape)
print(next(iter(trainloader)))

Files already downloaded and verified
Files already downloaded and verified
torch.Size([5, 1, 150, 150])
[tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]],


        [[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]],


        [[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,

Parameter optimization

In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 34 * 34, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # conv1 output width: input_width - (kernel_size - 1) => 150 - (5-1) = 146
        # pool 1 output width: int(input_width/2) => 73
        x = self.pool(F.relu(self.conv1(x)))
        # conv2 output width: input_width - (kernel_size - 1) => 73 - (5-1) = 69
        # pool 2 output width: int(input_width/2) => 34
        x = self.pool(F.relu(self.conv2(x)))  
        x = x.view(-1, 16 * 34 * 34)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
model = NeuralNetwork()

print_num = 10
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01)
for epoch in range(config.CLASSIFICATION_MODEL["num_epochs"]):  # loop over the dataset multiple times

    print(f'Epoch {epoch+1}')
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % print_num == (print_num-1):    # print every 50 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / print_num))
            running_loss = 0.0

print('Finished Training')



Epoch 1
[1,    10] loss: 1.130
[1,    20] loss: 0.707
[1,    30] loss: 0.694
[1,    40] loss: 0.783
[1,    50] loss: 0.700
[1,    60] loss: 0.697
[1,    70] loss: 0.698
[1,    80] loss: 0.709
[1,    90] loss: 0.693
[1,   100] loss: 0.695
[1,   110] loss: 0.688
[1,   120] loss: 0.720
[1,   130] loss: 0.998
Epoch 2
[2,    10] loss: 0.696
[2,    20] loss: 0.690
[2,    30] loss: 0.734
[2,    40] loss: 0.696
[2,    50] loss: 0.664
[2,    60] loss: 0.633
[2,    70] loss: 0.632
[2,    80] loss: 0.618
[2,    90] loss: 0.623
[2,   100] loss: 0.655
[2,   110] loss: 0.633
[2,   120] loss: 0.641
[2,   130] loss: 0.637
Epoch 3
[3,    10] loss: 0.527
[3,    20] loss: 0.499
[3,    30] loss: 0.691
[3,    40] loss: 0.487
[3,    50] loss: 0.597
[3,    60] loss: 0.452
[3,    70] loss: 0.513
[3,    80] loss: 0.578
[3,    90] loss: 0.537
[3,   100] loss: 0.636
[3,   110] loss: 0.952
[3,   120] loss: 0.501
[3,   130] loss: 0.492
Epoch 4
[4,    10] loss: 0.440
[4,    20] loss: 0.478
[4,    30] loss: 0.423
[4

In [7]:
dataiter = iter(testloader)
images, labels = next(dataiter)

print(labels)

output = model(images)
print(output)
_, predicted = torch.max(output, 1)
print(predicted)


tensor([1, 0, 1, 1, 1])
tensor([[-0.4004,  0.4409],
        [ 2.5882, -1.9237],
        [ 1.3276, -1.1502],
        [-2.2213,  2.1677],
        [-1.5250,  1.4255]], grad_fn=<AddmmBackward0>)
tensor([1, 0, 0, 1, 1])


In [8]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))

Accuracy of the network on the 88 test images: 83 %


Initialize model

Train

Test